# 데이터 전처리 - Model B (범주 일반화)
대학생 중도탈락(Dropout) 예측 - Target: Dropout(1) vs Non-Dropout(0, Graduate+Enrolled)

Model A(원본 구조 유지) 대비, 세분화된 범주형 값과 포르투갈 제도 특수성이 강한 값을
보편적으로 이해 가능한 상위 개념으로 일반화한 버전. 기준: `model_b_preprocessing_guide.html`

## 0. 라이브러리 & 데이터 로드

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib, json, os

RAW_PATH = "../data/raw/data.csv"
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(RAW_PATH, sep=';', encoding='utf-8-sig')
df.columns = [c.strip() for c in df.columns]
print("원본 shape:", df.shape)
shape_before = df.shape


원본 shape: (4424, 37)


## 1. Target 이진화

In [2]:
df['target'] = (df['Target'] == 'Dropout').astype(int)
df = df.drop(columns=['Target'])
print(df['target'].value_counts(normalize=True).round(3))


target
0    0.679
1    0.321
Name: proportion, dtype: float64


## 2. Application mode → 입학전형 유형 (Admission_pathway)
포르투갈 법령·지역·제도에 종속된 세부 전형(18종)을 8개 상위 개념으로 일반화.

In [3]:
admission_pathway_map = {
    1: '일반전형', 17: '일반전형', 18: '일반전형',
    5: '특별전형', 16: '특별전형',
    15: '외국인·국제학생 전형',
    39: '성인학습자 전형',
    42: '편입·전과', 43: '편입·전과', 51: '편입·전과', 57: '편입·전과',
    7: '고등교육 이수자 전형',
    44: '직업·기술교육 연계전형', 53: '직업·기술교육 연계전형',
    2: '기타·특수전형', 10: '기타·특수전형', 26: '기타·특수전형', 27: '기타·특수전형',
}
df['Admission_pathway'] = df['Application mode'].map(admission_pathway_map)

print("매핑 후 NaN 개수:", df['Admission_pathway'].isna().sum())
print("누락된 원본 코드:", df.loc[df['Admission_pathway'].isna(), 'Application mode'].unique())
print()
print(df['Admission_pathway'].value_counts(dropna=False))


매핑 후 NaN 개수: 0
누락된 원본 코드: []

Admission_pathway
일반전형            2704
성인학습자 전형         785
편입·전과            449
직업·기술교육 연계전형     248
고등교육 이수자 전형      139
특별전형              54
외국인·국제학생 전형       30
기타·특수전형           15
Name: count, dtype: int64


## 3. Course → 전공계열 (Major_field)
17개 개별 학과를 유사 분야끼리 상위 전공계열로 통합. (한국 공식 학과분류가 아니라 분석용 재분류)

In [4]:
major_field_map = {
    33: '공학·자연',       # Biofuel Production Technologies
    171: '예술·디자인',    # Animation and Multimedia Design
    8014: '사회',          # Social Service (evening attendance)
    9003: '자연·농생명',   # Agronomy
    9070: '예술·디자인',   # Communication Design
    9085: '보건',          # Veterinary Nursing
    9119: '공학·IT',       # Informatics Engineering
    9130: '자연·농생명',   # Equinculture
    9147: '경영',          # Management
    9238: '사회',          # Social Service
    9254: '경영·서비스',   # Tourism
    9500: '보건',          # Nursing
    9556: '보건',          # Oral Hygiene
    9670: '경영',          # Advertising and Marketing Management
    9773: '인문·사회',     # Journalism and Communication
    9853: '교육',          # Basic Education
    9991: '경영',          # Management (evening attendance)
}
df['Major_field'] = df['Course'].map(major_field_map)

print("매핑 후 NaN 개수:", df['Major_field'].isna().sum())
print("누락된 원본 코드:", df.loc[df['Major_field'].isna(), 'Course'].unique())
print()
print(df['Major_field'].value_counts(dropna=False))


매핑 후 NaN 개수: 0
누락된 원본 코드: []

Major_field
보건        1189
경영         916
사회         570
예술·디자인     441
자연·농생명     351
인문·사회      331
경영·서비스     252
교육         192
공학·IT      170
공학·자연       12
Name: count, dtype: int64


## 4. Previous qualification → 이전 학력 수준 (Previous_education_level)
세부 학력 코드(17종)를 학생이 이전에 어느 수준의 교육을 경험했는지 중심으로 통합.

In [5]:
prev_edu_map = {
    1: 'Secondary',
    2: 'Bachelor', 3: 'Bachelor', 40: 'Bachelor',
    4: 'Graduate', 5: 'Graduate', 43: 'Graduate',
    6: 'Higher education experience',
    9: 'Below secondary', 10: 'Below secondary', 12: 'Below secondary',
    14: 'Below secondary', 15: 'Below secondary', 19: 'Below secondary', 38: 'Below secondary',
    39: 'Vocational', 42: 'Vocational',
}
df['Previous_education_level'] = df['Previous qualification'].map(prev_edu_map)

print("매핑 후 NaN 개수:", df['Previous_education_level'].isna().sum())
print("누락된 원본 코드:", df.loc[df['Previous_education_level'].isna(), 'Previous qualification'].unique())
print()
print(df['Previous_education_level'].value_counts(dropna=False))


매핑 후 NaN 개수: 0
누락된 원본 코드: []

Previous_education_level
Secondary                      3717
Vocational                      255
Below secondary                 232
Bachelor                        189
Higher education experience      16
Graduate                         15
Name: count, dtype: int64


## 5. 부모 학력 → 교육 수준 (Mother/Father_education_level)
세부 교육과정 코드를 "중등교육 이하 → 고등학교 → 전문·직업교육 → 대학 → 대학원 → 미상·기타" 6단계로 통합.

In [6]:
parent_edu_map = {
    # 중등교육 이하
    9: '중등교육 이하', 10: '중등교육 이하', 11: '중등교육 이하', 12: '중등교육 이하',
    14: '중등교육 이하', 19: '중등교육 이하', 25: '중등교육 이하', 26: '중등교육 이하',
    27: '중등교육 이하', 29: '중등교육 이하', 30: '중등교육 이하', 37: '중등교육 이하', 38: '중등교육 이하',
    # 고등학교
    1: '고등학교', 13: '고등학교', 20: '고등학교',
    # 전문·직업교육
    18: '전문·직업교육', 22: '전문·직업교육', 31: '전문·직업교육', 33: '전문·직업교육',
    39: '전문·직업교육', 42: '전문·직업교육',
    # 대학
    2: '대학', 3: '대학', 6: '대학', 40: '대학',
    # 대학원
    4: '대학원', 5: '대학원', 41: '대학원', 43: '대학원', 44: '대학원',
    # 미상·기타
    34: '미상·기타', 35: '미상·기타', 36: '미상·기타',
}
df['Mother_education_level'] = df["Mother's qualification"].map(parent_edu_map)
df['Father_education_level'] = df["Father's qualification"].map(parent_edu_map)

for col in ['Mother_education_level', 'Father_education_level']:
    src = "Mother's qualification" if 'Mother' in col else "Father's qualification"
    print(f"[{col}] NaN 개수: {df[col].isna().sum()}, 누락코드: {df.loc[df[col].isna(), src].unique()}")

print()
print(df['Mother_education_level'].value_counts(dropna=False))
print()
print(df['Father_education_level'].value_counts(dropna=False))


[Mother_education_level] NaN 개수: 0, 누락코드: []
[Father_education_level] NaN 개수: 0, 누락코드: []

Mother_education_level
중등교육 이하    2590
고등학교       1069
대학          534
미상·기타       136
대학원          81
전문·직업교육      14
Name: count, dtype: int64

Father_education_level
중등교육 이하    2949
고등학교        906
대학          357
미상·기타       122
대학원          62
전문·직업교육      28
Name: count, dtype: int64


## 6. 부모 직업 → 직업군 (Mother/Father_occupation_group)
가이드 권고대로 먼저 실제 코드별 빈도를 확인한 뒤, "전문·관리직 / 사무·서비스 / 기술·생산 / 농림 / 무직·기타" 5개 직업군으로 통합.

In [7]:
print("=== Mother's occupation 빈도 ===")
print(df["Mother's occupation"].value_counts())
print()
print("=== Father's occupation 빈도 ===")
print(df["Father's occupation"].value_counts())


=== Mother's occupation 빈도 ===
Mother's occupation
9      1577
4       817
5       530
3       351
2       318
7       272
0       144
1       102
6        91
90       70
8        36
191      26
99       17
194      11
141       8
123       7
144       6
175       5
192       5
134       4
10        4
193       4
151       3
132       3
143       3
152       2
122       2
153       2
125       1
173       1
131       1
171       1
Name: count, dtype: int64

=== Father's occupation 빈도 ===
Father's occupation
9      1010
7       666
5       516
4       386
3       384
8       318
10      266
6       242
2       197
1       134
0       128
90       65
99       19
193      15
144       8
171       8
192       6
163       5
175       4
103       4
135       3
152       3
183       3
181       3
123       3
151       2
182       2
194       2
172       2
122       2
102       2
112       2
124       1
121       1
195       1
161       1
132       1
131       1
143       1
134       1
153    

In [8]:
occupation_group_map = {
    # 무직·기타
    0: '무직·기타', 90: '무직·기타', 99: '무직·기타',
    # 전문·관리직 (임원/전문직/기술직/군장교 등)
    1: '전문·관리직', 2: '전문·관리직', 3: '전문·관리직', 10: '전문·관리직',
    101: '전문·관리직', 102: '전문·관리직', 103: '전문·관리직',
    112: '전문·관리직', 114: '전문·관리직', 121: '전문·관리직', 122: '전문·관리직',
    123: '전문·관리직', 124: '전문·관리직', 125: '전문·관리직',
    131: '전문·관리직', 132: '전문·관리직', 134: '전문·관리직', 135: '전문·관리직',
    # 사무·서비스
    4: '사무·서비스', 5: '사무·서비스',
    141: '사무·서비스', 143: '사무·서비스', 144: '사무·서비스',
    151: '사무·서비스', 152: '사무·서비스', 153: '사무·서비스', 154: '사무·서비스', 195: '사무·서비스',
    # 기술·생산
    7: '기술·생산', 8: '기술·생산', 9: '기술·생산',
    171: '기술·생산', 172: '기술·생산', 173: '기술·생산', 174: '기술·생산', 175: '기술·생산',
    181: '기술·생산', 182: '기술·생산', 183: '기술·생산', 191: '기술·생산', 193: '기술·생산', 194: '기술·생산',
    # 농림
    6: '농림', 161: '농림', 163: '농림', 192: '농림',
}
df['Mother_occupation_group'] = df["Mother's occupation"].map(occupation_group_map)
df['Father_occupation_group'] = df["Father's occupation"].map(occupation_group_map)

for col in ['Mother_occupation_group', 'Father_occupation_group']:
    src = "Mother's occupation" if 'Mother' in col else "Father's occupation"
    missing = df.loc[df[col].isna(), src].unique()
    print(f"[{col}] NaN 개수: {df[col].isna().sum()}, 누락코드: {missing}")

print()
print(df['Mother_occupation_group'].value_counts(dropna=False))
print()
print(df['Father_occupation_group'].value_counts(dropna=False))


[Mother_occupation_group] NaN 개수: 0, 누락코드: []
[Father_occupation_group] NaN 개수: 0, 누락코드: []

Mother_occupation_group
기술·생산     1933
사무·서비스    1371
전문·관리직     793
무직·기타      231
농림          96
Name: count, dtype: int64

Father_occupation_group
기술·생산     2034
전문·관리직    1004
사무·서비스     920
농림         254
무직·기타      212
Name: count, dtype: int64


## 7. Nacionality → 중복 검토 후 제거
`International`(국제학생 여부) 변수와 실질적으로 겹치는 정보인지 확인.

In [9]:
cross = pd.crosstab(df['Nacionality'], df['International'])
print("Nacionality=1(Portuguese)이 아닌데 International=0인 경우:")
print(df[(df['Nacionality'] != 1) & (df['International'] == 0)].shape[0], "건")
print()
print("Nacionality 분포:")
print(df['Nacionality'].value_counts(normalize=True).round(4).head(5))


Nacionality=1(Portuguese)이 아닌데 International=0인 경우:
0 건

Nacionality 분포:
Nacionality
1     0.9751
41    0.0086
26    0.0032
6     0.0029
22    0.0029
Name: proportion, dtype: float64


In [10]:
df = df.drop(columns=['Nacionality'])
print("Nacionality 제거 완료. International 컬럼으로 국제학생 여부는 유지됨.")


Nacionality 제거 완료. International 컬럼으로 국제학생 여부는 유지됨.


## 8. 거시경제 변수 3개 제거
Unemployment rate / Inflation rate / GDP는 학생 개인 특성이 아니라 특정 시점의 포르투갈 경제상황을 나타내는 변수로,
국가·시점 의존성이 높아 일반화 전처리 모델에서는 제외.

In [11]:
df = df.drop(columns=['Unemployment rate', 'Inflation rate', 'GDP'])
print("거시경제 변수 3개 제거 완료:", df.shape)


거시경제 변수 3개 제거 완료: (4424, 40)


## 9. 파생변수 생성 (Model A와 동일하게 유지)
EDA에서 검증된 학업성취도/재정상태 파생변수는 Model B에서도 동일하게 활용.

In [12]:
df['sem1_approval_rate'] = np.where(df['Curricular units 1st sem (enrolled)'] > 0,
    df['Curricular units 1st sem (approved)'] / df['Curricular units 1st sem (enrolled)'], 0)
df['sem2_approval_rate'] = np.where(df['Curricular units 2nd sem (enrolled)'] > 0,
    df['Curricular units 2nd sem (approved)'] / df['Curricular units 2nd sem (enrolled)'], 0)
df['grade_change'] = df['Curricular units 2nd sem (grade)'] - df['Curricular units 1st sem (grade)']
df['zero_enrolled_1st_sem'] = (df['Curricular units 1st sem (enrolled)'] == 0).astype(int)
df['financial_risk_score'] = (
    (df['Tuition fees up to date'] == 0).astype(int) +
    (df['Debtor'] == 1).astype(int) +
    (df['Scholarship holder'] == 0).astype(int)
)
print("파생변수 5개 생성 완료")


파생변수 5개 생성 완료


## 10. 원본(일반화 이전) 컬럼 제거 + 전처리 전후 shape 검증

In [13]:
df = df.drop(columns=[
    'Application mode', 'Course', 'Previous qualification',
    "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation",
])

print("전처리 전:", shape_before)
print("전처리 후:", df.shape)
print()
print("결측치 확인:")
print(df.isna().sum()[df.isna().sum() > 0])


전처리 전: (4424, 37)
전처리 후: (4424, 38)

결측치 확인:
Series([], dtype: int64)


**검증 원칙**: 컬럼 수는 줄어들 수 있지만, 이유 없이 학생(row) 수가 줄어들면 안 된다 — row 수는 4,424로 그대로 유지되는지 확인.

## 11. 컬럼 분류 및 Train/Val/Test 분할

In [14]:
binary_cols = ['Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date',
               'Gender', 'Scholarship holder', 'International', 'Daytime/evening attendance']

low_card_categorical = ['Marital status', 'Admission_pathway', 'Major_field',
                         'Previous_education_level', 'Mother_education_level', 'Father_education_level',
                         'Mother_occupation_group', 'Father_occupation_group']

continuous_cols = ['Previous qualification (grade)', 'Admission grade', 'Age at enrollment',
                    'Curricular units 1st sem (grade)', 'Curricular units 2nd sem (grade)',
                    'Application order', 'sem1_approval_rate', 'sem2_approval_rate', 'grade_change']

count_cols = ['Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)',
              'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)',
              'Curricular units 1st sem (without evaluations)',
              'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)',
              'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)',
              'Curricular units 2nd sem (without evaluations)']

flag_cols = ['zero_enrolled_1st_sem', 'financial_risk_score']

feature_cols = binary_cols + low_card_categorical + continuous_cols + count_cols + flag_cols
print("총 사용 피처 수(인코딩 전):", len(feature_cols))
print("(참고: Model A는 41개였음 - Nacionality/거시경제3개 제거, 고카디널리티 6개가 저카디널리티 그룹 6개로 대체)")

X = df[feature_cols].copy()
y = df['target'].copy()

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(f"\ntrain={X_train.shape}, val={X_val.shape}, test={X_test.shape}")
print(f"dropout 비율 - train:{y_train.mean():.3f}, val:{y_val.mean():.3f}, test:{y_test.mean():.3f}")


총 사용 피처 수(인코딩 전): 37
(참고: Model A는 41개였음 - Nacionality/거시경제3개 제거, 고카디널리티 6개가 저카디널리티 그룹 6개로 대체)

train=(2654, 37), val=(885, 37), test=(885, 37)
dropout 비율 - train:0.321, val:0.322, test:0.321


## 12. 인코딩 & 스케일링
Model B는 모든 범주형 컬럼이 이미 저카디널리티(5~9종)로 일반화되어 있어, Model A에서 필요했던
"희소 카테고리 그룹화" 단계가 불필요함 — 일반화 자체가 그 역할을 겸함.

In [15]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), continuous_cols + count_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), low_card_categorical),
], remainder='passthrough')  # binary_cols, flag_cols

preprocessor.fit(X_train)

X_train_proc = preprocessor.transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

feature_names = list(preprocessor.get_feature_names_out())
print(f"최종 피처 개수: {len(feature_names)}")


최종 피처 개수: 81


## 13. 저장

In [16]:
pd.DataFrame(X_train_proc, columns=feature_names).assign(target=y_train.values).to_csv(f"{OUT_DIR}/train.csv", index=False)
pd.DataFrame(X_val_proc, columns=feature_names).assign(target=y_val.values).to_csv(f"{OUT_DIR}/val.csv", index=False)
pd.DataFrame(X_test_proc, columns=feature_names).assign(target=y_test.values).to_csv(f"{OUT_DIR}/test.csv", index=False)

joblib.dump(preprocessor, '../models/preprocessor.joblib')

schema = {
    "version": "Model B (범주 일반화)",
    "binary_cols": binary_cols,
    "low_card_categorical": low_card_categorical,
    "continuous_cols": continuous_cols,
    "count_cols": count_cols,
    "flag_cols": flag_cols,
    "dropped_columns": ["Nacionality", "Unemployment rate", "Inflation rate", "GDP",
                         "Application mode", "Course", "Previous qualification",
                         "Mother's qualification", "Father's qualification",
                         "Mother's occupation", "Father's occupation"],
    "final_feature_count": len(feature_names),
    "target_definition": "1=Dropout, 0=Non-Dropout(Graduate+Enrolled)",
}
with open(f"{OUT_DIR}/feature_schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

print("저장 완료: train.csv, val.csv, test.csv, preprocessor.joblib, feature_schema.json")


저장 완료: train.csv, val.csv, test.csv, preprocessor.joblib, feature_schema.json
